In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
# Load the merged dataset that includes `tsv_path`
merged_df_path = "final_clinical_survival_dataset2.csv"
merged_df = pd.read_csv(merged_df_path)

# Filter for non-null RNASeq paths
tsv_paths = merged_df["tsv_path"].dropna().unique()

# Container for gene expression data
expression_data = []
patient_ids = []

# Process each RNA-Seq file
for path in tqdm(tsv_paths[:4500]):  # limit for performance
    
    df = pd.read_csv(path, sep="\t", skiprows=1)
    df = df[['gene_id', 'fpkm_unstranded']]
    df.columns = ['gene_id', 'expression']
    df['expression'] = np.log2(df['expression'] + 1)

    # extract TCGA barcode (first 3 parts of the file name)
    barcode = os.path.basename(path).split('.')[0]
    patient_id = '-'.join(barcode.split('-')[:3])
    df['sample_id'] = patient_id

    expression_data.append(df)
    patient_ids.append(patient_id)
    

# Combine all expression data
print(f"Total files processed: {len(expression_data)} : {expression_data[0].shape if expression_data else 'No data'}")
combined_expression = pd.concat(expression_data)

# Pivot to create a matrix: rows=genes, columns=patients
expression_matrix = combined_expression.pivot(index="gene_id", columns="sample_id", values="expression")

# Remove genes with low variance
gene_variance = expression_matrix.var(axis=1)
high_var_genes = gene_variance[gene_variance > 0.5].index  # threshold may be tuned
expression_matrix_filtered = expression_matrix.loc[high_var_genes]

# Transpose for PCA (samples x genes)
X = expression_matrix_filtered.T.fillna(0)

# Normalize for PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)

# Convert to DataFrame
pca_df = pd.DataFrame(X_pca, index=X.index, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
pca_df["TCGA_Barcode"] = pca_df.index


# Save to file
pca_df.to_csv("D:/github_proj/cancer-survival-analysis/pca_gene_expression_data.csv", index=False)


C:\Users\Dr Kareem Kamal\AppData\Local\Temp\ipykernel_22124\552216943.py:9: DtypeWarning: Columns (42,43,44,45,47,49,51,52,53,54,55,57,58,59,60,61,62,63,64,65,67,68,69,78,79,81,82,83,85,86,87,88,89,91,92,93,94) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv(merged_df_path)
100%|██████████| 4500/4500 [09:37<00:00,  7.80it/s]


Total files processed: 4500 : (60664, 3)


: 

In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Load metadata with paths and barcodes
merged_df_path = "final_clinical_survival_dataset2.csv"
merged_df = pd.read_csv(merged_df_path)

# Filter for rows that have a valid tsv_path
valid_df = merged_df[merged_df["tsv_path"].notna()].copy()

# Map: tsv_path → TCGA_Barcode
path_to_barcode = dict(zip(valid_df["tsv_path"], valid_df["TCGA_Barcode"]))

# Prepare expression data
expression_data = []
patient_ids = []

for path in tqdm(path_to_barcode):
    if not os.path.exists(path):
        continue  # Skip if file doesn't exist
    
    df = pd.read_csv(path, sep="\t", skiprows=1)
    df = df[['gene_id', 'fpkm_unstranded']]
    df.columns = ['gene_id', 'expression']
    df['expression'] = np.log2(df['expression'] + 1)

    patient_id = path_to_barcode[path]
    df['sample_id'] = patient_id

    expression_data.append(df)
    patient_ids.append(patient_id)

# Combine expression data
combined_expression = pd.concat(expression_data)

# Pivot: genes as rows, patients as columns
expression_matrix = combined_expression.pivot(index="gene_id", columns="sample_id", values="expression")

# Filter low-variance genes
high_var_genes = expression_matrix.var(axis=1)[lambda x: x > 0.5].index
expression_matrix_filtered = expression_matrix.loc[high_var_genes]

# PCA processing
X = expression_matrix_filtered.T.fillna(0)
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, index=X.index, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
pca_df["TCGA_Barcode"] = pca_df.index

# Save output
pca_df.to_csv("pca_gene_expression_data_final.csv", index=False)


/tmp/ipykernel_197790/624382866.py:10: DtypeWarning: Columns (42,43,44,45,47,49,51,52,53,54,55,57,58,59,60,61,62,63,64,65,67,68,69,78,79,81,82,83,85,86,87,88,89,91,92,93,94) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv(merged_df_path)
 78%|███████▊  | 7251/9284 [12:39<03:33,  9.54it/s]   


KeyboardInterrupt: 

## Chunck for low Memory 

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Load metadata with paths and barcodes
merged_df_path = "final_clinical_survival_dataset2.csv"
merged_df = pd.read_csv(merged_df_path)
valid_df = merged_df[merged_df["tsv_path"].notna()].copy()
path_to_barcode = dict(zip(valid_df["tsv_path"], valid_df["TCGA_Barcode"]))
all_paths = list(path_to_barcode.keys())

# Output list for storing PCA results
pca_results = []

# Helper function to process a chunk of files
def process_chunk(tsv_paths_chunk, chunk_index):
    expression_data = []

    for path in tqdm(tsv_paths_chunk, desc=f"Processing chunk {chunk_index}"):
        if not os.path.exists(path):
            continue
        
        try:
            df = pd.read_csv(path, sep="\t", skiprows=1, usecols=["gene_id", "fpkm_unstranded"])
            df.columns = ['gene_id', 'expression']
            df['expression'] = np.log2(df['expression'] + 1)
            df['sample_id'] = path_to_barcode[path]
            expression_data.append(df)
        except Exception as e:
            print(f"Error in file {path}: {e}")
            continue

    if not expression_data:
        return None  # Skip empty chunk
    
    combined_expression = pd.concat(expression_data)

    # Ensure unique (gene_id, sample_id) combinations
    combined_expression = (
        combined_expression
        .groupby(["gene_id", "sample_id"], as_index=False)
        .agg({"expression": "mean"})  # You could also use "first"
    )

    # Now pivot safely
    expr_matrix = combined_expression.pivot(index="gene_id", columns="sample_id", values="expression")
    # Filter low-variance genes
    high_var_genes = expr_matrix.var(axis=1)[lambda x: x > 0.5].index
    expr_matrix_filtered = expr_matrix.loc[high_var_genes]

    # Transpose and fill NA
    X = expr_matrix_filtered.T.fillna(0)

    # Scale and apply PCA
    X_scaled = StandardScaler().fit_transform(X)
    max_components = min(X_scaled.shape[0], X_scaled.shape[1], 50)  # ≤ samples, features, 50
    pca = PCA(n_components=max_components)
    X_pca = pca.fit_transform(X_scaled)

    # Return dataframe
    chunk_pca_df = pd.DataFrame(X_pca, index=X.index, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
    chunk_pca_df["TCGA_Barcode"] = chunk_pca_df.index
    return chunk_pca_df

# Chunking logic
chunk_size = 50
chunks = [all_paths[i:i + chunk_size] for i in range(0, len(all_paths), chunk_size)]

# Process each chunk and collect results
for i, chunk in enumerate(chunks):
    result = process_chunk(chunk, i)
    if result is not None:
        pca_results.append(result)

# Final aggregation
final_pca_df = pd.concat(pca_results, axis=0)
final_pca_df.to_csv("pca_gene_expression_data2.csv", index=False)


C:\Users\Dr Kareem Kamal\AppData\Local\Temp\ipykernel_22124\3163389467.py:10: DtypeWarning: Columns (42,43,44,45,47,49,51,52,53,54,55,57,58,59,60,61,62,63,64,65,67,68,69,78,79,81,82,83,85,86,87,88,89,91,92,93,94) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv(merged_df_path)
Processing chunk 185: 100%|██████████| 34/34 [00:02<00:00, 15.71it/s]


In [6]:
merged_df.tsv_path[0]

'RNASeq_data/3396edba-0d2a-4485-ad0a-5114b38c1abe/2e64abe2-6024-4d28-9e09-560ce2a9fd15.rna_seq.augmented_star_gene_counts.tsv'

In [ ]:
df = pd.read_csv('RNASeq_data/3396edba-0d2a-4485-ad0a-5114b38c1abe/2e64abe2-6024-4d28-9e09-560ce2a9fd15.rna_seq.augmented_star_gene_counts.tsv',sep="\t")
df.columns 

Index(['# gene-model: GENCODE v36'], dtype='object')

In [19]:
import os

path = r'D:\github_proj\cancer-survival-analysis\RNASeq_data\3396edba-0d2a-4485-ad0a-5114b38c1abe\2e64abe2-6024-4d28-9e09-560ce2a9fd15.rna_seq.augmented_star_gene_counts.tsv'

print("Exists:", os.path.exists(path))


Exists: True


In [20]:
found = merged_df["tsv_path"].notna().sum()
print(f"✅ Found TSV files: {found} / {len(merged_df)}")


✅ Found TSV files: 0 / 9824


In [16]:
import os

def build_tsv_path(row):
    folder = os.path.join("D:/github_proj/cancer-survival-analysis/RNASeq_data", row["file_id"])
    file_name = row["File_Name"]
    full_path = os.path.normpath(os.path.join(folder, file_name))
    return full_path if os.path.exists(full_path) else None

merged_df["tsv_path"] = merged_df.apply(build_tsv_path, axis=1)


In [9]:
expression_data = []

for i, row in merged_df.iterrows():
    tsv_path = row["tsv_path"]
    if not os.path.exists(tsv_path):
        print(f"❌ File not found: {tsv_path}")
        continue
    try:
        df = pd.read_csv(tsv_path, sep="\t")
        # Add check to see if expected columns exist
        if "gene_id" not in df.columns or "FPKM" not in df.columns:
            print(f"⚠️ Missing expected columns in: {tsv_path}")
            continue
        df["expression"] = np.log2(df["FPKM"] + 1)
        df["sample_id"] = row["patient_id"]
        expression_data.append(df[["sample_id", "gene_id", "expression"]])
    except Exception as e:
        print(f"❌ Error reading {tsv_path}: {e}")
        continue

if not expression_data:
    print("⚠️ No expression data was loaded. Check your paths and files.")


⚠️ Missing expected columns in: D:\github_proj\cancer-survival-analysis\RNASeq_data\3396edba-0d2a-4485-ad0a-5114b38c1abe\2e64abe2-6024-4d28-9e09-560ce2a9fd15.rna_seq.augmented_star_gene_counts.tsv
⚠️ Missing expected columns in: D:\github_proj\cancer-survival-analysis\RNASeq_data\fb32f0b1-4fb7-43d5-8091-dc13a1f6d9e8\1d468785-141d-40ca-acb1-d6d85a8c9d7b.rna_seq.augmented_star_gene_counts.tsv
⚠️ Missing expected columns in: D:\github_proj\cancer-survival-analysis\RNASeq_data\de5e449b-bc57-4836-89e4-73a3ba24abdf\748e4eaa-2b96-4dce-a903-c7df733d7f50.rna_seq.augmented_star_gene_counts.tsv
⚠️ Missing expected columns in: D:\github_proj\cancer-survival-analysis\RNASeq_data\a58e2ae3-a236-4209-8292-70465216cb85\e614fbb4-7574-4704-9525-c0aea4c10fc6.rna_seq.augmented_star_gene_counts.tsv
⚠️ Missing expected columns in: D:\github_proj\cancer-survival-analysis\RNASeq_data\f7b5d569-1a46-4f23-bfca-a46daacda793\ad8c55ae-46df-43cd-8cb5-c049e3019959.rna_seq.augmented_star_gene_counts.tsv
⚠️ Missing expe

KeyboardInterrupt: 